# Notebook 05: Extreme Rainfall Frequency Analysis
**GEV return levels and flood threshold exceedance for Jakarta Greater Capital Region**

**Models**: MRI-ESM2-0, EC-Earth3, CNRM-CM6-1  
**Scenarios**: historical, SSP1-2.6, SSP2-4.5, SSP5-8.5  
**Variables**: Rx1day, Rx3day, Rx5day  
**Distribution**: GEV (MLE, scipy sign convention corrected to hydrology convention)

> **Prerequisite:** Run `extreme_frequency.py --model all --scenario all` before executing this notebook.

## 1. Setup

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
import matplotlib.lines as mlines, matplotlib.patches as mpatches
import matplotlib.cm as _cm
import matplotlib.colors as _colors

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy import stats          # used by fit_gev, gev_return_level
from scipy import stats as _stats  # alias used in bootstrap cell
from scipy.interpolate import RegularGridInterpolator

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Paths
EXTREME_DIR = Path('../../py/results/extreme_freq')
INDICES_DIR = Path('../../py/results/indices')
RESULTS     = Path('../../py/results/figures')
TABLES      = Path('../../py/results/tables')
RESULTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# ✔️CMIP6 registry
MODELS = {
    'MRI-ESM2-0': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#2364a5',
    },
    'EC-Earth3': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#e07b00',
    },
    'CNRM-CM6-1': {
        'ensemble':  'r1i1p1f2',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#7b2d8b',
    },
}

ALL_SCENARIOS = ['historical', 'ssp126', 'ssp245', 'ssp585']
SSP_SCENARIOS = ['ssp126', 'ssp245', 'ssp585']
SCENARIO_COLORS = {
    'historical': '#555555',
    'ssp126':     '#2166AC',
    'ssp245':     '#4DAF4A',
    'ssp585':     '#D73027',
}
SCENARIO_LABELS = {
    'historical': 'Historical',
    'ssp126':     'SSP1-2.6',
    'ssp245':     'SSP2-4.5',
    'ssp585':     'SSP5-8.5',
}
MODEL_LINESTYLES = {'MRI-ESM2-0': '-', 'EC-Earth3': '--', 'CNRM-CM6-1': ':'}

HIST_PERIOD = (1950, 2014)
FAR_PERIOD  = (2071, 2100)
CENTER_LAT, CENTER_LON = -7.5, 106.875

RETURN_PERIODS = [2, 5, 10, 25, 50, 100]
EXT_VARS       = ['Rx1day', 'Rx3day', 'Rx5day']

JAKARTA_FLOOD_THRESHOLDS = {
    'Moderate (100 mm)': 100.0,
    'Severe (150 mm)':   150.0,
    'Extreme (200 mm)':  200.0,
}

print('Setup complete ✔️')

In [ ]:
# lobal font settings
PLOT_FONT        = 'Calibri'
PLOT_FONT_ALT    = ['Arial', 'Times New Roman']

_available = {f.name for f in _fm.fontManager.ttflist}
_font_to_use = PLOT_FONT if PLOT_FONT in _available else next(
    (f for f in PLOT_FONT_ALT if f in _available), 'Times New Roman'
)
mpl.rcParams['font.family'] = _font_to_use
mpl.rcParams['axes.titlesize']  = 11
mpl.rcParams['axes.labelsize']  = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['legend.fontsize'] = 9

print(f'Font set to: {_font_to_use}')
if _font_to_use != PLOT_FONT:
    print(f'  (Note: {PLOT_FONT!r} not found — using {_font_to_use!r} instead)')

In [ ]:
print('=' * 45)
for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    for scenario in cfg['scenarios']:
        for var in EXT_VARS:
            fname = f'{model}_{scenario}_{ensemble}_{var}_extreme_freq_jakarta.nc'
            status = '✔️'
            print(f'  {status}  {model:<15} {scenario:<12} {var}')
print('=' * 45)

### Configs

In [ ]:
from scipy import stats  # GEV fitting

def load_extreme(model, scenario, var):
    """
    Load GEV extreme frequency Dataset for one model × scenario × variable.
    Returns Dataset or None.
    """
    cfg      = MODELS[model]
    ensemble = cfg['ensemble']
    fname    = f'{model}_{scenario}_{ensemble}_{var}_extreme_freq_jakarta.nc'
    fpath    = EXTREME_DIR / fname
    if not fpath.exists():
        return None
    return xr.open_dataset(
        fpath,
        decode_times=xr.coders.CFDatetimeCoder(use_cftime=True)
    )


def load_ams(model, scenario, var, indices_dir=INDICES_DIR):
    """
    Load Annual Maximum Series from the precipitation indices output.
    Returns 1-D array at the centre cell or None.
    """
    cfg      = MODELS[model]
    ensemble = cfg['ensemble']
    fname    = f'{model}_{scenario}_{ensemble}_indices_jakarta.nc'
    fpath    = indices_dir / fname
    if not fpath.exists():
        return None, None
    ds  = xr.open_dataset(fpath, decode_times=xr.coders.CFDatetimeCoder(use_cftime=True))
    ams = ds[var].sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest').values.astype(float)
    yrs = ds['year'].values.astype(int)
    ds.close()
    valid = np.isfinite(ams) & (ams > 0)
    return ams[valid], yrs[valid]



def fit_gev(ams, var='unknown', verbose=True):
    """
    Fit GEV (MLE). scipy uses negative shape; convert to hydrology convention.

    Degenerate fit detection: rejects fits where the 100-yr return level is
    physically implausible given the variable:
      - Rx1day:  RL100 must be in [10, 2000] mm
      - Rx3day:  RL100 must be in [20, 5000] mm
      - Rx5day:  RL100 must be in [30, 8000] mm
      - default: RL100 must be in [5, 5000] mm

    Returns None for degenerate fits; prints the RL100 value when verbose=True.
    """
    RL100_LIMITS = {
        'Rx1day': (10, 2000),
        'Rx3day': (20, 5000),
        'Rx5day': (30, 8000),
    }
    lo, hi = RL100_LIMITS.get(var, (5, 5000))

    try:
        shape_scipy, loc, scale = stats.genextreme.fit(ams)
    except Exception as e:
        if verbose:
            print(f'    [fit_gev] MLE failed for {var}: {e}')
        return None

    shape = -shape_scipy   # hydrology sign convention

    try:
        rl100 = gev_return_level(loc, scale, shape, 100)
    except Exception as e:
        if verbose:
            print(f'    [fit_gev] RL100 computation failed for {var}: {e}')
        return None

    if not np.isfinite(rl100) or rl100 < lo or rl100 > hi:
        if verbose:
            print(f'    [fit_gev] Degenerate: {var} RL100={rl100:.1f} mm '
                  f'(expected [{lo}, {hi}]), '
                  f'shape={shape:.4f}, AMS mean={ams.mean():.1f}, n={len(ams)}')
        return None

    return {'loc': loc, 'scale': scale, 'shape': shape}

def gev_return_level(loc, scale, shape, T):
    """Return level for return period T."""
    p  = 1 - 1.0 / T
    yT = -np.log(p)
    if abs(shape) < 1e-6:
        return loc - scale * np.log(yT)
    return loc - (scale / shape) * (1 - yT ** (-shape))


def exceedance_prob(loc, scale, shape, threshold):
    """Annual exceedance probability for a GEV-fitted variable."""
    if abs(shape) < 1e-6:
        p_ne = np.exp(-np.exp(-(threshold - loc) / scale))
    else:
        z = 1 + shape * (threshold - loc) / scale
        if z <= 0:
            return 1.0
        p_ne = np.exp(-(z ** (-1.0 / shape)))
    return float(1 - p_ne)




def plot_spatial(da, title, cmap, units, output_path=None, diverging=False):
    """Plot a 2-D (lat × lon) DataArray as a smooth Cartopy map."""
    from scipy.interpolate import RegularGridInterpolator

    # Display parameters
    MAP_FONT        = 'Calibri'      # preferred font; falls back if not installed
    MAP_FONT_ALT    = ['Arial', 'Times New Roman']  # fallback chain
    FIG_WIDTH       = 9              # figure width in inches
    FIG_HEIGHT      = 5              # figure height in inches
    UPSAMPLE_N      = 300            # interpolation grid resolution (N × N)
    COASTLINE_WIDTH = 1.0
    BORDER_WIDTH    = 0.4
    GRIDLINE_ALPHA  = 0.5
    COLORBAR_SHRINK = 0.85
    TITLE_FONTSIZE  = 11
    DPI             = 150
    OCEAN_COLOR     = '#d0e8f0'

    # Apply font preference
    import matplotlib.font_manager as _fm
    _available = {f.name for f in _fm.fontManager.ttflist}
    _font = MAP_FONT if MAP_FONT in _available else next(
        (f for f in MAP_FONT_ALT if f in _available), 'Times New Roman'
    )
    plt.rcParams['font.family'] = _font

    lons = da.lon.values
    lats = da.lat.values
    vals = da.values.astype(float)

    # Colour scale from coarse data
    finite = vals[np.isfinite(vals)]
    if diverging:
        vmax = float(np.abs(finite).max()) if len(finite) else 1.0
        vmin_plot, vmax_plot = -vmax, vmax
    else:
        vmin_plot = float(finite.min()) if len(finite) else 0.0
        vmax_plot = float(finite.max()) if len(finite) else 1.0

    # Upsample: RegularGridInterpolator requires ascending lat axis
    lat_asc   = lats if lats[0] < lats[-1] else lats[::-1]
    vals_asc  = vals if lats[0] < lats[-1] else vals[::-1, :]

    interp = RegularGridInterpolator(
        (lat_asc, lons), vals_asc,
        method='linear',
        bounds_error=False,
        fill_value=np.nan,
    )
    lons_fine = np.linspace(lons.min(), lons.max(), UPSAMPLE_N)
    lats_fine = np.linspace(lat_asc.min(), lat_asc.max(), UPSAMPLE_N)
    grid_lon, grid_lat = np.meshgrid(lons_fine, lats_fine)
    vals_up = interp((grid_lat, grid_lon))

    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    # OCEAN background
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR, zorder=1)

    # Data: imshow on the upsampled array with bicubic smoothing on top
    ax.imshow(
        vals_up,
        extent=[lons.min(), lons.max(), lat_asc.min(), lat_asc.max()],
        origin='lower',
        cmap=cmap,
        vmin=vmin_plot, vmax=vmax_plot,
        interpolation='bicubic',
        transform=ccrs.PlateCarree(),
        zorder=2,
        aspect='auto',
    )

    # Dummy pcolormesh for the colorbar (imshow colorbar works but pcolormesh
    # gives a cleaner scalar mappable with the exact same vmin/vmax)
    _norm = _colors.Normalize(vmin=vmin_plot, vmax=vmax_plot)
    _sm   = plt.cm.ScalarMappable(cmap=cmap, norm=_norm)
    _sm.set_array([])
    plt.colorbar(_sm, ax=ax, label=units, shrink=COLORBAR_SHRINK, pad=0.02)

    ax.add_feature(cfeature.COASTLINE, linewidth=COASTLINE_WIDTH, zorder=5)
    ax.add_feature(cfeature.BORDERS,   linewidth=BORDER_WIDTH, linestyle=':', zorder=4)

    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray',
                      alpha=GRIDLINE_ALPHA, linestyle='--')
    gl.top_labels   = False
    gl.right_labels = False

    ax.set_extent([104.125, 109.375, -7.5, -5], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    return fig

print('Helpers defined ✔️')

## 2. Annual Maximum Series using Descriptive Statistics

Quick overview of the AMS at the Jakarta centre cell by model and scenario.

In [ ]:
print(f'AMS Statistics at Jakarta Centre Cell (lat={CENTER_LAT}, lon={CENTER_LON})')
print('=' * 72)
print(f'  {"Model":<15} {"Scenario":<12} {"Var":<8} {"n":>4} {"Mean":>8} {"Std":>8} {"Max":>8}')
print('-' * 72)

for model in MODELS:
    for scenario in ALL_SCENARIOS:
        if scenario not in MODELS[model]['scenarios']:
            continue
        for var in EXT_VARS:
            ams, yrs = load_ams(model, scenario, var)
            if ams is None:
                continue
            print(f'  {model:<15} {SCENARIO_LABELS[scenario]:<12} {var:<8} '
                  f'{len(ams):>4} {ams.mean():>8.1f} {ams.std():>8.1f} {ams.max():>8.1f}')

## 3. GEV Fitting

GEV parameters (μ location, σ scale, ξ shape) fitted by MLE at the Jakarta centre cell.  
Positive shape (ξ > 0) = heavy tail (Fréchet); ξ < 0 = bounded tail (Weibull).

In [ ]:
gev_params = {}   # {(model, scenario, var): {loc, scale, shape}}

print(f'{"Model":<15} {"Scenario":<12} {"Var":<8} {"μ (loc)":>10} {"σ (scale)":>10} {"ξ (shape)":>10}')
print('-' * 72)

for model in MODELS:
    for scenario in ALL_SCENARIOS:
        if scenario not in MODELS[model]['scenarios']:
            continue
        for var in EXT_VARS:
            ams, _ = load_ams(model, scenario, var)
            if ams is None or len(ams) < 5:
                print(f'{model:<15} {SCENARIO_LABELS[scenario]:<12} {var:<8} '
                      f'  [no AMS data or insufficient samples]')
                continue
            # Pass var name so fit_gev uses variable-specific RL100 limits
            p = fit_gev(ams, var=var, verbose=True)
            if p is None:
                print(f'{model:<15} {SCENARIO_LABELS[scenario]:<12} {var:<8} '
                      f'  [degenerate fit — skipped]')
                continue
            gev_params[(model, scenario, var)] = p
            print(f'{model:<15} {SCENARIO_LABELS[scenario]:<12} {var:<8} '
                  f'{p["loc"]:>10.2f} {p["scale"]:>10.2f} {p["shape"]:>10.4f}')


## 4. GEV Goodness-of-Fit on QQ-Plots (Historical)

Theoretical vs empirical quantiles for the historical AMS at the centre cell.  
Points should lie close to the 1:1 line if GEV is a good fit.

In [ ]:
fig, axes = plt.subplots(len(MODELS), len(EXT_VARS),
                          figsize=(5 * len(EXT_VARS), 4 * len(MODELS)))

for ri, model in enumerate(MODELS):
    for ci, var in enumerate(EXT_VARS):
        ax  = axes[ri][ci]
        key = (model, 'historical', var)
        ams, _ = load_ams(model, 'historical', var)
        if ams is None or key not in gev_params or gev_params[key] is None:
            ax.set_title(f'{model}\n{var} — [degenerate/no fit]', fontsize=8)
            ax.set_visible(False)
            continue

        p          = gev_params[key]
        ams_sorted = np.sort(ams)
        n          = len(ams_sorted)
        emp_prob   = (np.arange(1, n + 1) - 0.44) / (n + 0.12)   # Gringorten
        theo_q     = stats.genextreme.ppf(emp_prob, -p['shape'],
                                           loc=p['loc'], scale=p['scale'])
        ax.scatter(theo_q, ams_sorted, s=18, alpha=0.8, color=MODELS[model]['color'])
        lim = max(theo_q.max(), ams_sorted.max()) * 1.05
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2)
        ax.set_xlabel('GEV Theoretical Quantiles [mm]')
        ax.set_ylabel('Observed AMS [mm]')
        ax.set_title(f'{model}\n{var} (Historical GEV QQ)', fontweight='bold', fontsize=9)
        ax.grid(True, alpha=0.3)

plt.suptitle('GEV Goodness-of-Fit on Historical AMS, Jakarta Centre Cell',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '05_gev_qqplot_historical.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Return Level Curves on All Models × Scenarios

Return level curves (log-scale x-axis) by model and scenario for each variable.  
Models = linestyle; scenarios = colour.

In [ ]:
T_fine = np.logspace(np.log10(1.1), np.log10(200), 100)

for var in EXT_VARS:
    fig, ax = plt.subplots(figsize=(9, 5))

    for model in MODELS:
        ls = MODEL_LINESTYLES[model]
        for scenario in ALL_SCENARIOS:
            key = (model, scenario, var)
            if key not in gev_params:
                continue
            p  = gev_params[key]
            rl = [gev_return_level(p['loc'], p['scale'], p['shape'], T) for T in T_fine]
            ax.plot(T_fine, rl, linestyle=ls, linewidth=1.8,
                    color=SCENARIO_COLORS[scenario],
                    alpha=0.85)

    # Mark standard return periods
    for T in [10, 50, 100]:
        ax.axvline(T, color='gray', linewidth=0.6, linestyle=':', alpha=0.5)

    ax.set_xscale('log')
    ax.set_xlabel('Return Period [years]')
    ax.set_ylabel(f'{var} [mm]')
    ax.set_title(f'Return Level Curves | {var}\nAll models × scenarios, Jakarta Centre Cell',
                 fontweight='bold')
    ax.grid(True, alpha=0.4)

    # Build legend
    scen_leg  = [mpatches.Patch(color=SCENARIO_COLORS[s], label=SCENARIO_LABELS[s])
                 for s in ALL_SCENARIOS]
    model_leg = [mlines.Line2D([], [], color='grey', linestyle=MODEL_LINESTYLES[m], label=m)
                 for m in MODELS]
    ax.legend(handles=scen_leg + model_leg, fontsize=8, ncol=2)

    plt.tight_layout()
    plt.savefig(RESULTS / f'05_return_levels_{var}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print summary table
    print(f'\nReturn Levels [mm] | {var}  (centre cell, ensemble mean per scenario):')
    print(f'  {"Scenario":<14}' + ''.join([f'{T:>8d}yr' for T in RETURN_PERIODS]))
    for scenario in ALL_SCENARIOS:
        rls_all = []
        for model in MODELS:
            key = (model, scenario, var)
            if key not in gev_params:
                continue
            p = gev_params[key]
            rls_all.append([gev_return_level(p['loc'], p['scale'], p['shape'], T)
                            for T in RETURN_PERIODS])
        if not rls_all:
            continue
        ens_rl = np.mean(rls_all, axis=0)
        print(f'  {SCENARIO_LABELS[scenario]:<14}' + ''.join([f'{v:>8.1f}' for v in ens_rl]))

## 6. Flood Threshold Exceedance Probabilities

Annual probability of exceeding Jakarta's operational flood thresholds under each scenario.  
Ensemble mean ± inter-model range.

In [ ]:
exc_rows = []

for scenario in ALL_SCENARIOS:
    for label, thresh in JAKARTA_FLOOD_THRESHOLDS.items():
        probs = []
        for model in MODELS:
            key = (model, scenario, 'Rx1day')
            if key not in gev_params:
                continue
            p = gev_params[key]
            probs.append(exceedance_prob(p['loc'], p['scale'], p['shape'], thresh))
        if not probs:
            continue
        ens_mean = np.mean(probs)
        ens_std  = np.std(probs)
        rp       = 1 / ens_mean if ens_mean > 0 else np.inf
        exc_rows.append({
            'Scenario':        SCENARIO_LABELS[scenario],
            'Threshold':       label,
            'Prob (mean)':     round(ens_mean, 4),
            'Prob (std)':      round(ens_std,  4),
            'Return period':   round(rp, 1),
        })

df_exc = pd.DataFrame(exc_rows)
print('Annual Exceedance Probabilities | Rx1day GEV, Jakarta Centre Cell')
print('(Ensemble mean ± inter-model std across 3 CMIP6 models)')
print(df_exc.to_string(index=False))
df_exc.to_csv(TABLES / 'flood_exceedance_probabilities.csv', index=False)
print('\nSaved → py/results/tables/flood_exceedance_probabilities.csv')

In [ ]:
# Bar chart: exceedance probability by scenario × threshold
thresholds  = list(JAKARTA_FLOOD_THRESHOLDS.keys())
n_thresh    = len(thresholds)
n_scenarios = len(ALL_SCENARIOS)
bar_width   = 0.18
x           = np.arange(n_thresh)

fig, ax = plt.subplots(figsize=(10, 5))

for i, scenario in enumerate(ALL_SCENARIOS):
    probs_per_thresh = []
    for label in thresholds:
        row = df_exc[(df_exc['Scenario'] == SCENARIO_LABELS[scenario]) &
                     (df_exc['Threshold'] == label)]
        probs_per_thresh.append(float(row['Prob (mean)'].values[0]) if len(row) else 0)

    offset = (i - (n_scenarios - 1) / 2) * bar_width
    ax.bar(x + offset, probs_per_thresh, width=bar_width,
           color=SCENARIO_COLORS[scenario], label=SCENARIO_LABELS[scenario],
           alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(thresholds, fontsize=9)
ax.set_ylabel('Annual Exceedance Probability')
ax.set_title('Jakarta Flood Threshold Exceedance Probabilities\n'
             'Rx1day GEV | 3-model ensemble mean',
             fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(RESULTS / '05_flood_exceedance_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Spatial Return Level Maps

10-year and 100-year return levels of Rx1day from the GEV output files, for historical and SSP5-8.5.

In [ ]:
for T_yr in [10, 100]:
    for scenario in ['historical', 'ssp585']:
        rl_maps  = []
        lons_ref = lats_ref = None

        for model in MODELS:
            ds = load_extreme(model, scenario, 'Rx1day')
            if ds is None:
                continue
            key = f'Rx1day_RL{T_yr}yr'
            if key not in ds:
                continue

            m_lats = ds.lat.values
            m_lons = ds.lon.values
            m_vals = ds[key].values.astype(float)

            # Use the first model's grid as the common reference
            if lons_ref is None:
                lons_ref = m_lons
                lats_ref = m_lats
                rl_maps.append(m_vals)
            else:
                # Grids differ between models (gn vs gr)
                if m_vals.shape == rl_maps[0].shape and \
                   np.allclose(m_lats, lats_ref, atol=0.1) and \
                   np.allclose(m_lons, lons_ref, atol=0.1):
                    rl_maps.append(m_vals)
                else:
                    # Interpolate this model onto the reference grid
                    lat_asc = m_lats if m_lats[0] < m_lats[-1] else m_lats[::-1]
                    val_asc = m_vals if m_lats[0] < m_lats[-1] else m_vals[::-1, :]
                    interp  = RegularGridInterpolator(
                        (lat_asc, m_lons), val_asc,
                        method='linear', bounds_error=False, fill_value=np.nan,
                    )
                    ref_lat_asc = lats_ref if lats_ref[0] < lats_ref[-1] else lats_ref[::-1]
                    gg_lon, gg_lat = np.meshgrid(lons_ref, ref_lat_asc)
                    remapped = interp((gg_lat, gg_lon))
                    # Re-orient to match lats_ref direction
                    if lats_ref[0] > lats_ref[-1]:
                        remapped = remapped[::-1, :]
                    rl_maps.append(remapped)

        if not rl_maps:
            print(f'⚠ No data for Rx1day RL{T_yr}yr / {scenario}')
            continue

        ens_rl = np.nanmean(np.stack(rl_maps, axis=0), axis=0)
        da_rl  = xr.DataArray(ens_rl,
                               coords={'lat': lats_ref, 'lon': lons_ref},
                               dims=['lat', 'lon'])

        fig = plot_spatial(
            da_rl,
            title=f'Rx1day | {T_yr}-Year Return Level  |  {SCENARIO_LABELS[scenario]}\n'
                  f'3-model ensemble mean  |  GEV MLE',
            cmap='YlOrRd',
            units='mm',
            output_path=RESULTS / f'05_Rx1day_RL{T_yr}yr_{scenario}_ensemble.png',
        )
        plt.show()

## Bootstrap Confidence Intervals on Return Levels

MLE-fitted GEV parameters carry estimation uncertainty due to the finite AMS record (65 years historical). Parametric bootstrap quantifies this: synthetic AMS samples are drawn from the fitted GEV, refitted, and return levels recomputed 1000×.

The 95% CI is the 2.5–97.5 percentile range of bootstrapped return levels.

In [ ]:
# Bootstrap confidence intervals on return levels
#
# GEV parameters are estimated by MLE on N=30–65 years of AMS data, which
# introduces substantial estimation uncertainty. Bootstrap CIs quantify this.
#
# Method: parametric bootstrap resample N synthetic AMS values from the
# fitted GEV, refit parameters, compute return level. Repeat 1000×.
# The 2.5–97.5 percentile range is the 95% CI on each return level.
#
# Editable parameters
RL_BOOTSTRAP_N  = 1000      # bootstrap resamples (increase for publication)
RL_CI           = 0.95      # confidence interval level
RL_T_PLOT       = [10, 50, 100]  # return periods (years) to report CIs for
RL_VARS         = ['Rx1day']     # variables to bootstrap

print('Bootstrap 95% CIs on GEV Return Levels on Historical AMS, Jakarta Centre Cell')
print(f'N bootstrap = {RL_BOOTSTRAP_N}')
print()

alpha_tail = (1 - RL_CI) / 2
rng = np.random.default_rng(42)

for var in RL_VARS:
    print(f'Variable: {var}')
    print(f'  {"Model":<15} ' +
          '  '.join(f'RL{T:>3}yr [CI]' for T in RL_T_PLOT))
    print('  ' + '-' * 70)

    for model in MODELS:
        ams, _ = load_ams(model, 'historical', var)
        if ams is None or len(ams) < 10:
            print(f'  {model:<15}  insufficient AMS data')
            continue

        p = fit_gev(ams)
        if p is None:
            print(f'  {model:<15}  degenerate GEV fit — skipped')
            continue

        # MLE central estimate
        rl_central = {T: gev_return_level(p['loc'], p['scale'], p['shape'], T)
                      for T in RL_T_PLOT}

        # Parametric bootstrap
        boot_rls = {T: [] for T in RL_T_PLOT}
        for _ in range(RL_BOOTSTRAP_N):
            # Sample synthetic AMS from the fitted GEV
            # scipy genextreme uses negative shape convention
            syn = _stats.genextreme.rvs(-p['shape'], loc=p['loc'],
                                         scale=p['scale'], size=len(ams),
                                         random_state=rng)
            syn = syn[syn > 0]   # physical constraint
            if len(syn) < 5:
                continue
            try:
                sh_b, lo_b, sc_b = _stats.genextreme.fit(syn)
                p_b = {'loc': lo_b, 'scale': sc_b, 'shape': -sh_b}
                # Apply the same degenerate-fit guard as fit_gev():
                # discard this bootstrap draw if 100-yr RL is implausible
                rl100_b = gev_return_level(p_b['loc'], p_b['scale'], p_b['shape'], 100)
                if not (np.isfinite(rl100_b) and 0 < rl100_b < 5000):
                    continue
                for T in RL_T_PLOT:
                    rl_b = gev_return_level(p_b['loc'], p_b['scale'], p_b['shape'], T)
                    if np.isfinite(rl_b) and 0 < rl_b < 5000:
                        boot_rls[T].append(rl_b)
            except Exception:
                continue

        row = f'  {model:<15}'
        for T in RL_T_PLOT:
            if boot_rls[T]:
                ci_lo = np.percentile(boot_rls[T], alpha_tail * 100)
                ci_hi = np.percentile(boot_rls[T], (1 - alpha_tail) * 100)
                central = rl_central[T]
                row += f'  {central:.0f} [{ci_lo:.0f}–{ci_hi:.0f}]'
            else:
                row += f'  {rl_central[T]:.0f} [CI unavailable]'
        print(row)

    print()

##### ✅ Summary

**Key findings:**
- 100-year return levels of Rx1day increase substantially under SSP5-8.5 across all three CMIP6 models
- Exceedance probability for the 150 mm/day severe flood threshold increases under all SSP scenarios
- The GEV shape parameter (ξ) is positive for all models at the Jakarta centre cell, indicating a heavy-tailed Fréchet distribution, which mean extremes are more likely than a Gumbel (ξ=0) assumption would suggest
- Inter-model spread in return levels is largest for the longest return periods (50–100 year), as expected

**Files saved:**
```
py/results/tables/
  flood_exceedance_probabilities.csv
py/results/figures/
  05_gev_qqplot_historical.png
  05_return_levels_{Rx1day, Rx3day, Rx5day}.png
  05_flood_exceedance_bars.png
  05_Rx1day_RL{10,100}yr_{historical,ssp585}_ensemble.png
```

**Next:** `06_erosivity_ml.ipynb`